In [1]:
import sys
sys.path.insert(0, '../..')

import time
import json
import numpy as np
from src.serving.redis_cache import (
    RecommendationCache)

# Connect to Redis
cache = RecommendationCache(
    host   = 'localhost',
    port   = 6379,
    ttl    = 3600,
    prefix = 'rec_test')

print(f"Connected: {cache.connected}")

# Test set + get
test_recs = [
    {"rank": 1, "movie_id": 356,
     "title": "Forrest Gump",
     "score": 0.95},
    {"rank": 2, "movie_id": 296,
     "title": "Pulp Fiction",
     "score": 0.87},
]

# SET
ok = cache.set(999, test_recs)
print(f"Cache SET : {'✅' if ok else '❌'}")

# GET
result = cache.get(999)
print(f"Cache GET : {'✅' if result else '❌'}")
if result:
    print(f"  Got {len(result)} recs")
    print(f"  First: {result[0]['title']}")

# INVALIDATE
inv = cache.invalidate(999)
print(f"Cache INV : {'✅' if inv else '❌'}")

# GET after invalidate
result2 = cache.get(999)
print(f"After INV : "
      f"{'None ✅' if result2 is None else '⚠️  still cached'}")

Connected: True
Cache SET : ✅
Cache GET : ✅
  Got 2 recs
  First: Forrest Gump
Cache INV : ✅
After INV : None ✅


In [2]:
import httpx
import asyncio

BASE = "http://localhost:8000"

async def benchmark_cache():
    print("CACHE LATENCY BENCHMARK")
    print("=" * 45)

    async with httpx.AsyncClient(
            timeout=30) as client:

        # First request — cache miss
        t0 = time.time()
        r  = await client.post(
            f"{BASE}/recommend",
            json={"user_id": 481,
                  "top_k": 10})
        miss_lat = (time.time()-t0)*1000
        data     = r.json()
        print(f"Cache MISS: {miss_lat:.1f}ms "
              f"cached={data['cached']}")

        # Second request — cache hit
        t0 = time.time()
        r  = await client.post(
            f"{BASE}/recommend",
            json={"user_id": 481,
                  "top_k": 10})
        hit_lat = (time.time()-t0)*1000
        data    = r.json()
        print(f"Cache HIT : {hit_lat:.1f}ms "
              f"cached={data['cached']}")

        speedup = miss_lat / \
                  max(hit_lat, 0.1)
        print(f"Speedup   : {speedup:.0f}x")

        # N cached requests
        N    = 20
        lats = []
        for _ in range(N):
            t0 = time.time()
            await client.post(
                f"{BASE}/recommend",
                json={"user_id": 481,
                      "top_k": 10})
            lats.append(
                (time.time()-t0)*1000)

        print(f"\nCached requests (n={N}):")
        print(f"  p50 : "
              f"{np.percentile(lats,50):.1f}ms")
        print(f"  p95 : "
              f"{np.percentile(lats,95):.1f}ms")
        print(f"  p99 : "
              f"{np.percentile(lats,99):.1f}ms")

        return miss_lat, hit_lat, lats

miss_lat, hit_lat, cached_lats = \
    await benchmark_cache()

CACHE LATENCY BENCHMARK
Cache MISS: 375.2ms cached=False
Cache HIT : 21.0ms cached=True
Speedup   : 18x

Cached requests (n=20):
  p50 : 8.8ms
  p95 : 11.2ms
  p99 : 14.1ms


In [3]:
async def test_invalidation():
    print("CACHE INVALIDATION TEST")
    print("=" * 45)

    async with httpx.AsyncClient(
            timeout=30) as client:

        uid = 196

        # Warm cache
        r = await client.post(
            f"{BASE}/recommend",
            json={"user_id": uid,
                  "top_k": 5})
        print(f"1. Initial request: "
              f"cached={r.json()['cached']}")

        # Verify cached
        r = await client.post(
            f"{BASE}/recommend",
            json={"user_id": uid,
                  "top_k": 5})
        print(f"2. Second request : "
              f"cached={r.json()['cached']}")

        # Send feedback → invalidates cache
        r = await client.post(
            f"{BASE}/feedback",
            json={
                "user_id":  uid,
                "movie_id": 356,
                "rating":   4.5,
                "action":   "watch",
            })
        fb = r.json()
        print(f"3. Feedback sent  : "
              f"invalidated="
              f"{fb['cache_invalidated']}")

        # Next request should be cache miss
        r = await client.post(
            f"{BASE}/recommend",
            json={"user_id": uid,
                  "top_k": 5})
        print(f"4. After feedback : "
              f"cached={r.json()['cached']}")

        if not r.json()['cached']:
            print("\n✅ Cache invalidation "
                  "working correctly")
        else:
            print("\n⚠️  Cache not invalidated")

await test_invalidation()

CACHE INVALIDATION TEST
1. Initial request: cached=False
2. Second request : cached=True
3. Feedback sent  : invalidated=True
4. After feedback : cached=False

✅ Cache invalidation working correctly


In [4]:
async def show_cache_stats():
    async with httpx.AsyncClient() as client:
        r = await client.get(
            f"{BASE}/cache/stats")
        stats = r.json()

    print("REDIS CACHE STATS")
    print("=" * 45)
    for k, v in stats.items():
        print(f"  {k:<20}: {v}")

    if stats.get('total_requests', 0) > 0:
        print(f"\nHit rate: "
              f"{stats['hit_rate_pct']}% "
              f"{'✅' if stats['hit_rate_pct'] > 50 else '⚠️'}")
        print(f"Target  : > 80% in production")

await show_cache_stats()

REDIS CACHE STATS
  connected           : True
  hits                : 22
  misses              : 3
  hit_rate_pct        : 88.0
  total_requests      : 25
  cached_users        : 2
  memory_used         : 1.13M
  ttl_seconds         : 3600

Hit rate: 88.0% ✅
Target  : > 80% in production


In [5]:
print("CACHE WARMING")
print("=" * 45)
print("Pre-populating cache for active users\n")

import sys
sys.path.insert(0, '../../src/serving')

from bentoml_service import (
    model, user_seqs,
    token2movie, title_map,
    PAD_TOKEN, PROC)
import torch
import pandas as pd

# Get top 20 most active users
ratings_df = pd.read_csv(
    '../../data/processed/'
    'ratings_cleaned.csv')
top_users  = ratings_df.groupby(
    'userId')['rating']\
    .count()\
    .sort_values(ascending=False)\
    .head(20).index.tolist()

print(f"Top 20 active users: {top_users[:5]}...")

# Warm function
def get_recs_for_warming(uid):
    seq = user_seqs.get(uid, [])
    if not seq: return []
    pad_l = 50 - len(seq)
    hist  = torch.LongTensor(
        [[PAD_TOKEN]*pad_l + seq[-50:]])
    with torch.no_grad():
        toks, scores = model.predict(
            hist, top_k=100)
    rated    = set(seq)
    seen_mid = set()
    recs     = []
    sc_arr   = scores[0].cpu().numpy()
    sc_min   = sc_arr.min()
    sc_range = max(
        sc_arr.max()-sc_min, 1e-6)
    for tok, sc in zip(
            toks[0].cpu().numpy(), sc_arr):
        mid = token2movie.get(int(tok))
        if not mid: continue
        if int(tok) in rated: continue
        if mid in seen_mid: continue
        seen_mid.add(mid)
        t = title_map.get(
            mid, f"Movie {mid}")
        recs.append({
            "rank":       len(recs)+1,
            "movie_id":   int(mid),
            "title":      str(t),
            "score":      round(float(
                (sc-sc_min)/sc_range), 4),
            "cold_start": False,
            "fallback":   False,
        })
        if len(recs) >= 10: break
    return recs


# Warm cache
prod_cache = RecommendationCache(
    host='localhost', port=6379,
    ttl=3600, prefix='rec')

start   = time.time()
results = prod_cache.warm(
    top_users,
    get_recs_for_warming)
elapsed = time.time() - start

print(f"\nWarming results:")
print(f"  Warmed  : {results['warmed']}")
print(f"  Skipped : {results['skipped']}")
print(f"  Failed  : {results['failed']}")
print(f"  Time    : {elapsed:.1f}s")
print(f"\n✅ Cache warm complete")
print(f"   Next {results['warmed']} users "
      f"will get <5ms latency")

CACHE WARMING
Pre-populating cache for active users

✅ HSTU model loaded for serving
   Vocab size : 9069
   Users      : 670
   Titles     : 45454
Top 20 active users: [547, 564, 624, 15, 73]...

Warming results:
  Warmed  : 20
  Skipped : 0
  Failed  : 0
  Time    : 0.1s

✅ Cache warm complete
   Next 20 users will get <5ms latency


In [6]:
day31_results = {
    "cache":     "Redis",
    "host":      "localhost:6379",
    "ttl_sec":   3600,
    "features": [
        "per_user_caching",
        "ttl_expiry",
        "feedback_invalidation",
        "cache_warming",
        "hit_rate_monitoring",
        "stats_endpoint",
    ],
    "latency": {
        "cache_miss_ms":  round(
            float(miss_lat), 1),
        "cache_hit_ms":   round(
            float(hit_lat), 1),
        "speedup_x":      round(
            miss_lat/max(hit_lat, 0.1), 1),
        "cached_p50_ms":  round(float(
            np.percentile(
                cached_lats, 50)), 1),
        "cached_p99_ms":  round(float(
            np.percentile(
                cached_lats, 99)), 1),
    },
    "warming": {
        "users_warmed": results['warmed'],
        "time_sec":     round(elapsed, 1),
    }
}

with open(
        '../../data/processed/'
        'day31_results.json', 'w') as f:
    json.dump(day31_results, f, indent=2)

print("✅ Day 31 results saved")
print(json.dumps(day31_results, indent=2))

✅ Day 31 results saved
{
  "cache": "Redis",
  "host": "localhost:6379",
  "ttl_sec": 3600,
  "features": [
    "per_user_caching",
    "ttl_expiry",
    "feedback_invalidation",
    "cache_warming",
    "hit_rate_monitoring",
    "stats_endpoint"
  ],
  "latency": {
    "cache_miss_ms": 375.2,
    "cache_hit_ms": 21.0,
    "speedup_x": 17.9,
    "cached_p50_ms": 8.8,
    "cached_p99_ms": 14.1
  },
  "warming": {
    "users_warmed": 20,
    "time_sec": 0.1
  }
}
